# Notebook principal — INF6243

Ce notebook exécute le pipeline complet et affiche les résultats clés.

In [1]:
from pathlib import Path
from time import perf_counter
from IPython.display import display, Markdown, Image

import sys

sys.path.insert(0, "Code")
from notebook_workflow import (
    build_models_table,
    build_runs_comparison_table,
    load_report,
    run_all_configs,
)
from result_interpreter import interpret_report
import run_configs

In [2]:
import os, glob
base = "/home/kilo/Work/Cours - UQO/technique-d-apprentissage/Tech-App-Devoir-II/.venv-gpu/lib/python3.11/site-packages/nvidia"
for p in [
    f"{base}/cuda_nvrtc/lib",
    f"{base}/cuda_runtime/lib",
    f"{base}/cusolver/lib",
    f"{base}/cublas/lib",
    f"{base}/cusparse/lib",
]:
    print(p, "exists:", os.path.isdir(p), "files:", len(glob.glob(p+"/*")))
    

/home/kilo/Work/Cours - UQO/technique-d-apprentissage/Tech-App-Devoir-II/.venv-gpu/lib/python3.11/site-packages/nvidia/cuda_nvrtc/lib exists: True files: 4
/home/kilo/Work/Cours - UQO/technique-d-apprentissage/Tech-App-Devoir-II/.venv-gpu/lib/python3.11/site-packages/nvidia/cuda_runtime/lib exists: True files: 3
/home/kilo/Work/Cours - UQO/technique-d-apprentissage/Tech-App-Devoir-II/.venv-gpu/lib/python3.11/site-packages/nvidia/cusolver/lib exists: True files: 4
/home/kilo/Work/Cours - UQO/technique-d-apprentissage/Tech-App-Devoir-II/.venv-gpu/lib/python3.11/site-packages/nvidia/cublas/lib exists: True files: 5
/home/kilo/Work/Cours - UQO/technique-d-apprentissage/Tech-App-Devoir-II/.venv-gpu/lib/python3.11/site-packages/nvidia/cusparse/lib exists: True files: 3


In [3]:
import os
base = "/home/kilo/Work/Cours - UQO/technique-d-apprentissage/Tech-App-Devoir-II/.venv-gpu/lib/python3.11/site-packages/nvidia"
paths = [
    f"{base}/cuda_nvrtc/lib",
    f"{base}/cuda_runtime/lib",
    f"{base}/cusolver/lib",
    f"{base}/cublas/lib",
    f"{base}/cusparse/lib",
]
os.environ["LD_LIBRARY_PATH"] = ":".join(paths) + ":" + os.environ.get("LD_LIBRARY_PATH","")
print(os.environ["LD_LIBRARY_PATH"])

/home/kilo/Work/Cours - UQO/technique-d-apprentissage/Tech-App-Devoir-II/.venv-gpu/lib/python3.11/site-packages/nvidia/cuda_nvrtc/lib:/home/kilo/Work/Cours - UQO/technique-d-apprentissage/Tech-App-Devoir-II/.venv-gpu/lib/python3.11/site-packages/nvidia/cuda_runtime/lib:/home/kilo/Work/Cours - UQO/technique-d-apprentissage/Tech-App-Devoir-II/.venv-gpu/lib/python3.11/site-packages/nvidia/cusolver/lib:/home/kilo/Work/Cours - UQO/technique-d-apprentissage/Tech-App-Devoir-II/.venv-gpu/lib/python3.11/site-packages/nvidia/cublas/lib:/home/kilo/Work/Cours - UQO/technique-d-apprentissage/Tech-App-Devoir-II/.venv-gpu/lib/python3.11/site-packages/nvidia/cusparse/lib:/home/kilo/Work/Cours - UQO/technique-d-apprentissage/Tech-App-Devoir-II/.venv-gpu/lib/python3.11/site-packages/nvidia/cuda_runtime/lib:/home/kilo/Work/Cours - UQO/technique-d-apprentissage/Tech-App-Devoir-II/.venv-gpu/lib/python3.11/site-packages/nvidia/cuda_nvrtc/lib:/home/kilo/Work/Cours - UQO/technique-d-apprentissage/Tech-App-Dev

In [4]:
import ctypes, cupy as cp
ctypes.CDLL("libnvrtc.so.12")
print("nvrtc OK")
cp.show_config()

nvrtc OK
OS                           : Linux-6.19.11-1-cachyos-x86_64-with-glibc2.43
Python Version               : 3.11.14
CuPy Version                 : 14.0.1
CuPy Platform                : NVIDIA CUDA
NumPy Version                : 2.2.6
SciPy Version                : 1.14.1
Cython Build Version         : 3.1.4
Cython Runtime Version       : None
CUDA Root                    : /opt/cuda
NVCC PATH                    : /opt/cuda/bin/nvcc
CUDA Build Version           : 12090
CUDA Driver Version          : 13020
CUDA Runtime Version         : 12090 (linked to CuPy) / 12040 (locally installed)
CUDA Extra Include Dirs      : ['/home/kilo/Work/Cours - UQO/technique-d-apprentissage/Tech-App-Devoir-II/.venv-gpu/lib/python3.11/site-packages/nvidia/cuda_runtime/include']
cuBLAS Version               : (available)
cuFFT Version                : 11201
cuRAND Version               : 10305
cuSOLVER Version             : (11, 6, 1)
cuSPARSE Version             : (available)
NVRTC Version         

In [5]:
import models
import preprocessing as prep
from pathlib import Path
df = prep.clean_data(prep.load_data(Path("Data/labeled_data.csv")))
x = df["clean_tweet"]
y = df["class"]
x_train, x_val, x_test, y_train, y_val, y_test = prep.train_val_test_split(x, y, test_size=0.2, val_size=0.1, random_state=42)
res = models.train_all_models(
    x_train=x_train, y_train=y_train, x_val=x_val, y_val=y_val,
    random_state=42, include_distilbert=False,
    algorithm_switches={"KNNGPU": False, "LinearSVCGPU": True, "LogisticRegressionGPU": False, "RandomForestGPU": False},
    cv_folds=3, scoring="f1_macro",
)
print({k: (v["status"], v["error"]) for k, v in res.items()})

{'NaiveBayes': ('trained', None), 'LogisticRegression': ('trained', None), 'LinearSVC': ('trained', None), 'KNN': ('trained', None), 'DecisionTree': ('trained', None), 'RandomForest': ('trained', None), 'AdaBoost': ('trained', None), 'MLPClassifier': ('trained', None), 'LinearSVCGPU': ('failed', '\nAll the 72 fits failed.\nIt is very likely that your model is misconfigured.\nYou can try to debug the error by setting error_score=\'raise\'.\n\nBelow are more details about the failures:\n--------------------------------------------------------------------------------\n1 fits failed with the following error:\nTraceback (most recent call last):\n  File "/home/kilo/Work/Cours - UQO/technique-d-apprentissage/Tech-App-Devoir-II/.venv-gpu/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score\n    estimator.fit(X_train, y_train, **fit_params)\n  File "/home/kilo/Work/Cours - UQO/technique-d-apprentissage/Tech-App-Devoir-II/.venv-gpu/lib/python3.11/site

## Paramètres du notebook (à modifier)

Cette section regroupe **tous les paramètres utilisateur**. Le détail des profils et des grilles est centralisé dans `Code/run_configs.py`.

- `RUN_MATRIX`
  - Valeurs: `"default"` ou `"exhaustive"`.
  - `default`: matrice courte (itérations plus rapides).
  - `exhaustive`: matrice complète (plus de runs/profils, plus coûteux).

- `SELECTED_MODELS`
  - Liste des modèles à exécuter **directement depuis le notebook**.
  - Utiliser les noms de `run_configs.BASE_MODEL_NAMES`.
  - Exemple: `("DistilBERT", "LinearSVC", "KNNGPU")`.
  - Le notebook filtre automatiquement les runs pour ne garder que ces modèles.

- `DISTILBERT_PROXY_PENALTY`
  - Malus appliqué lors de la comparaison inter-runs si DistilBERT est évalué avec un CV proxy.
  - Recommandé: `0.00` à `0.05`.
  - Plus la valeur est élevée, plus on pénalise les runs DistilBERT quand la CV complète n'est pas effectuée.

- `GPU_MODELS_ENABLED`
  - Modèles GPU cuML activés (`LogisticRegressionGPU`, `LinearSVCGPU`, `KNNGPU`, `RandomForestGPU`).
  - Permet d'activer/désactiver des modèles GPU sans toucher le code source.

- `GPU_PROFILES_ENABLED`
  - Profils de tuning GPU (`gpu_fast`, `gpu_balanced`, `gpu_aggressive`).
  - `gpu_fast`: rapide, peu de combinaisons.
  - `gpu_balanced`: compromis coût/performance.
  - `gpu_aggressive`: recherche large, plus coûteuse.

- `DISTILBERT_PROFILES_ENABLED`
  - Profils DistilBERT (`fast`, `balanced`, `robust`, `vram_max`).
  - Impact principal: epochs, batch size, max length, VRAM/temps.

- `MLP_PROFILES_ENABLED` / `ADABOOST_PROFILES_ENABLED`
  - Profils de tuning classiques.
  - Plus les profils sont "larges", plus le GridSearchCV est long.

### Conseils pratiques

- Pour un test rapide: garder `RUN_MATRIX="default"`.
- Pour une étude complète: `RUN_MATRIX="exhaustive"` + profils ciblés.
- Pour lancer uniquement certains modèles: modifier `SELECTED_MODELS`.
- Si la machine manque de mémoire/VRAM: retirer `gpu_aggressive` et `vram_max`.
- Si cuML/DistilBERT n'est pas disponible, les runs incompatibles sont filtrés automatiquement.

In [ ]:
# Paramètres utilisateur uniquement (la logique est centralisée dans `Code/run_configs.py`).

# Choix de la matrice de runs:
# - "default": exécution rapide
# - "exhaustive": couverture complète des profils
RUN_MATRIX = "exhaustive"  # "default" ou "exhaustive"

# Sélection directe des modèles à lancer depuis le notebook.
#
# Modèles disponibles (noms exacts à utiliser):
# - NaiveBayes
# - LogisticRegression
# - LinearSVC
# - KNN
# - DecisionTree
# - RandomForest
# - AdaBoost
# - MLPClassifier
# - LogisticRegressionGPU
# - LinearSVCGPU
# - KNNGPU
# - RandomForestGPU
# - DistilBERT
#
# Comment sélectionner / ne pas sélectionner:
# - Tous les modèles: SELECTED_MODELS = tuple(AVAILABLE_MODELS)
# - Inclure seulement certains modèles: SELECTED_MODELS = ("LinearSVC", "KNN", "DistilBERT")
# - Exclure certains modèles: SELECTED_MODELS = tuple(m for m in AVAILABLE_MODELS if m not in ("DistilBERT", "KNNGPU"))
# - Aucun modèle (déconseillé): une erreur explicite est levée.
#
# Important: les noms sont sensibles à la casse et doivent correspondre exactement.
AVAILABLE_MODELS = tuple(run_configs.BASE_MODEL_NAMES)
# SELECTED_MODELS = tuple(AVAILABLE_MODELS)
# Exemples:
# SELECTED_MODELS = ("DistilBERT",)
# SELECTED_MODELS = ("LinearSVC", "KNN", "MLPClassifier")
SELECTED_MODELS = ("LinearSVCGPU", "KNNGPU", "RandomForestGPU", "LogisticRegressionGPU")

# Malus de prudence pour les runs DistilBERT évalués avec CV proxy.
# Recommandé: 0.00 à 0.05.
DISTILBERT_PROXY_PENALTY = run_configs.DISTILBERT_PROXY_PENALTY_DEFAULT

# Options disponibles (profils/modèles exposés par run_configs).
RUN_OPTIONS = run_configs.get_available_exhaustive_options()
# Modèles GPU activés (retirer un nom pour désactiver).
GPU_MODELS_ENABLED = tuple(RUN_OPTIONS["gpu_models"])

# Profils GPU activés: fast (rapide), balanced (compromis), aggressive (plus coûteux).
# GPU_PROFILES_ENABLED = ("gpu_fast", "gpu_balanced", "gpu_aggressive")
GPU_PROFILES_ENABLED = ("gpu_fast",)

# Profils activés par famille de modèles.
DISTILBERT_PROFILES_ENABLED = tuple(RUN_OPTIONS["distilbert_profiles"])
MLP_PROFILES_ENABLED = tuple(RUN_OPTIONS["mlp_profiles"])
ADABOOST_PROFILES_ENABLED = tuple(RUN_OPTIONS["adaboost_profiles"])

selected_set = set(SELECTED_MODELS)
unknown_models = sorted(selected_set.difference(AVAILABLE_MODELS))
if unknown_models:
    raise ValueError(
        "Modèles inconnus dans SELECTED_MODELS: "
        f"{unknown_models}. Disponibles: {AVAILABLE_MODELS}"
    )

# Construction des runs à exécuter + filtrage automatique des runs incompatibles
# (dépendances manquantes pour DistilBERT ou cuML).
RUNS, SKIPPED_RUNS = run_configs.build_active_runs(
    run_matrix=RUN_MATRIX,
    include_baseline=True,
    distilbert_profile_names=DISTILBERT_PROFILES_ENABLED,
    mlp_profile_names=MLP_PROFILES_ENABLED,
    adaboost_profile_names=ADABOOST_PROFILES_ENABLED,
    gpu_profile_names=GPU_PROFILES_ENABLED,
    gpu_model_names=GPU_MODELS_ENABLED,
)
if SKIPPED_RUNS:
    print("Runs ignorés (dépendances absentes):", ", ".join(SKIPPED_RUNS))

# Filtrage final des runs selon SELECTED_MODELS.
FILTERED_RUNS = {}
for run_name, run_ctx in RUNS.items():
    run_cfg = dict(run_ctx["config"])
    run_switches = run_cfg.get("algorithm_switches", {})
    if not isinstance(run_switches, dict):
        continue

    filtered_switches = {
        model_name: bool(enabled) and (model_name in selected_set)
        for model_name, enabled in run_switches.items()
    }
    if not any(filtered_switches.values()):
        continue

    run_cfg["algorithm_switches"] = filtered_switches
    run_cfg["include_distilbert"] = bool(filtered_switches.get("DistilBERT", False))
    FILTERED_RUNS[run_name] = {
        "why": run_ctx["why"] + " [filtré via SELECTED_MODELS]",
        "config": run_cfg,
    }

RUNS = FILTERED_RUNS
if not RUNS:
    raise ValueError(
        "Aucun run actif après filtrage par SELECTED_MODELS. "
        "Ajustez la sélection de modèles ou les profils activés."
    )

print("Profils actifs:")
print("- DistilBERT:", DISTILBERT_PROFILES_ENABLED)
print("- MLP:", MLP_PROFILES_ENABLED)
print("- AdaBoost:", ADABOOST_PROFILES_ENABLED)
print("- GPU models:", GPU_MODELS_ENABLED)
print("- GPU profiles:", GPU_PROFILES_ENABLED)
print("- Models sélectionnés:", SELECTED_MODELS)
print("- Runs actifs:", tuple(RUNS.keys()))

# Exécute la matrice de runs et mesure le temps total notebook.
t0 = perf_counter()
workflow = run_all_configs(RUNS, distilbert_proxy_penalty=DISTILBERT_PROXY_PENALTY)
print(f"[timing] notebook_run_all_configs: {perf_counter() - t0:.2f}s")
run_summary_df = workflow["run_summary_df"]
BEST_RUN = workflow["best_run"]
artifacts = workflow["artifacts"]
FIGURE_NAMES = workflow["figure_names"]

# Affichage synthétique des scores inter-runs.
display(
    run_summary_df[
        [
            "run",
            "best_model",
            "include_distilbert",
            "distilbert_cv_proxy",
            "fairness_penalty",
            "best_selection_score",
            "adjusted_selection_score",
            "best_test_f1_macro",
        ]
    ]
)
print(f"Run de reference selectionne (score ajusté): {BEST_RUN}")
artifacts

## Pourquoi une matrice explicite de runs ?

- **Objectif**: chercher la meilleure configuration de chaque modèle complexe séparément.
- **Couverture**: 1 run = 1 profil (`DistilBERT`, `MLPClassifier`, `AdaBoost`, et 4 modèles GPU), plus un baseline.
- **Approche non conservative**: profils mémoire/VRAM agressifs (batch size, max_length, grilles larges) pour pousser la recherche.
- **Affinage ensuite**: après ce balayage, conserver les meilleurs profils par modèle et raffiner localement.
- **Comparaison équitable**: même politique de scoring, avec compensation DistilBERT CV proxy via `DISTILBERT_PROXY_PENALTY`.
- **Maintenance**: la matrice est centralisée dans `Code/run_configs.py` et partagée entre notebook et CLI.

Cette stratégie évite le produit cartésien coûteux et garde une lecture claire des effets de chaque famille de paramètres.

In [ ]:
t0 = perf_counter()
report = load_report(Path(artifacts["report_path"]))
print(f"[timing] notebook_load_report: {perf_counter() - t0:.2f}s")

print("Meilleur modèle:", report.get("best_model"))
print("Métriques test:", report.get("best_model_test_metrics"))
print("DistilBERT:", report.get("distilbert_note"))
print("Configuration:", report.get("run_config"))
display(Markdown("**Justification:** " + report.get("best_model_selection_explanation", "")))

phase_times = report.get("timing_seconds", {})
if phase_times:
    print("\nTemps d'exécution par phase (pipeline):")
    for phase_name, seconds in sorted(phase_times.items(), key=lambda item: item[1], reverse=True):
        print(f"- {phase_name}: {seconds:.2f}s")

In [ ]:
# Tableau complet: tous les modèles, statut, métriques, hyperparamètres
build_models_table(report)

In [ ]:
# Interpreteur de resultats (script dedie)
interpreter_summary = interpret_report(report, weak_f1_threshold=0.55)
interpreter_summary

## Pourquoi `best_cv_score` peut être vide pour DistilBERT ?

- Les modèles classiques utilisent `GridSearchCV`, donc un `best_cv_score` est produit.
- DistilBERT est entraîné en fine-tuning direct (pas de GridSearchCV complet pour limiter le coût).
- Le fallback de robustesse est documenté dans `model_selection_method.cv_fallback_for_models`.

In [ ]:
# Comparaison inter-runs (avec compensation DistilBERT CV proxy)
reports_dir = Path(artifacts["report_path"]).parent
comparison_df = build_runs_comparison_table(
    reports_dir,
    distilbert_proxy_penalty=DISTILBERT_PROXY_PENALTY,
)

if comparison_df.empty:
    print("Aucun report multi-run detecte.")
else:
    ordered_cols = [
        "run",
        "best_model",
        "include_distilbert",
        "distilbert_cv_proxy",
        "fairness_penalty",
        "best_selection_score",
        "adjusted_selection_score",
        "best_test_f1_macro",
        "distilbert_note",
    ]
    display(comparison_df[ordered_cols])

    print("\n=== Ligue classique (sans DistilBERT) ===")
    classic_df = comparison_df[comparison_df["include_distilbert"] == False]
    if classic_df.empty:
        print("Aucun run classique trouvé.")
    else:
        display(classic_df[ordered_cols])

    print("\n=== Ligue étendue (avec DistilBERT) ===")
    distil_df = comparison_df[comparison_df["include_distilbert"] == True]
    if distil_df.empty:
        print("Aucun run DistilBERT trouvé.")
    else:
        display(distil_df[ordered_cols])

    print("Astuce: la comparaison est aussi disponible en figure (runs_comparison_overview.png).")

In [ ]:
fig_dir = Path(artifacts["figures_dir"])
for fig_name in FIGURE_NAMES:
    fig_path = fig_dir / fig_name
    if fig_path.exists():
        display(Markdown(f"### {fig_name}"))
        display(Image(filename=str(fig_path)))
    else:
        print(f"Figure absente: {fig_path}")